# 문제 3: Tesseract를 활용한 약봉투 이미지 문자 인식(OCR) 실습

**소재**: 약봉투 이미지 (한글 + 영어 혼용 텍스트)  
**목표**: pytesseract로 약품명·복약안내·주의사항 텍스트 추출  
**가산점 포인트**:
- `lang='kor+eng'` 옵션으로 한글+영어 혼용 추출
- 전처리 파이프라인(흑백 변환 → 업스케일 → 이진화 → 노이즈 제거)으로 인식률 향상
- 전체 이미지 배치 처리
- 약봉투 특화 키워드 하이라이팅

## 1. 환경 설정 — Tesseract 설치 및 라이브러리 임포트

In [ ]:
import os, subprocess

# Tesseract OCR 엔진 + 한국어 학습 데이터 설치
!apt-get install -y tesseract-ocr tesseract-ocr-kor > /dev/null 2>&1

# Python 라이브러리 설치
!pip install pytesseract pillow opencv-python-headless -q

# TESSDATA_PREFIX 동적 탐지 (Tesseract 버전에 무관)
result = subprocess.run(
    ['find', '/usr/share/tesseract-ocr', '-name', 'kor.traineddata'],
    capture_output=True, text=True
)
kor_path = result.stdout.strip().split('\n')[0]
if not kor_path:
    raise RuntimeError("kor.traineddata 미설치 — apt-get 단계를 다시 확인하세요.")
os.environ['TESSDATA_PREFIX'] = os.path.dirname(kor_path)
os.environ['TESSERACT_LANG']  = 'kor+eng'   # 한글+영어 혼용

# Tesseract OCR 설정
# PSM 11: sparse text (방향·순서 무관) — 약봉투처럼 텍스트가 흩어진 이미지에 최적
# OEM 1:  LSTM only — 한국어는 LSTM 모델만 지원하므로 반드시 1로 설정
#         (OEM 3은 legacy 엔진을 섞어 쓰는데 legacy는 한국어 미지원이라
#          한글이 알파벳으로 잘못 인식되는 문제가 발생)
os.environ['TESSERACT_CONFIG'] = '--psm 11 --oem 1'

print("설치 완료")
print(f"TESSDATA_PREFIX  : {os.environ['TESSDATA_PREFIX']}")
print(f"TESSERACT_LANG   : {os.environ['TESSERACT_LANG']}")
print(f"TESSERACT_CONFIG : {os.environ['TESSERACT_CONFIG']}")
print()
print("── 설치된 언어 확인 (kor 포함되어야 함) ──")
!tesseract --list-langs
print()
!tesseract --version

In [ ]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm

# 나눔고딕 폰트 설치 후 캐시 초기화 (설치만으로는 적용 안 됨)
!apt-get install -y fonts-nanum > /dev/null 2>&1
fm._load_fontmanager(try_read_cache=False)

nanum_path = fm.findfont(fm.FontProperties(family='NanumGothic'))
if 'NanumGothic' not in nanum_path:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    fm.fontManager.addfont(font_path)
    nanum_path = font_path

matplotlib.rc('font', family='NanumGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

print("라이브러리 임포트 완료")
print(f"적용된 한글 폰트: {nanum_path}")

## 2. 이미지 다운로드 — URL 리스트에서 자동 수집

학습 서버에 업로드된 약봉투 이미지를 파일명 리스트로 지정하여 다운로드합니다.

In [ ]:
import urllib.request

# 이미지가 호스팅된 베이스 URL
BASE_URL = 'https://rmp4.learningfactory.co.kr/mobileContent/test/'

# 처리할 이미지 파일명 리스트 (필요한 파일만 추가)
FILENAMES = [
    'med_1.jpg',
    'med_2.jpeg',
    'med_3.jpg',
    'med_4.jpg',
    'med_5.jpg',
    'med_6.jpeg',
    'med_7.jpeg',
    'med_8.jpeg',
    'med_9.jpeg',
    'med_10.jpeg',
    'med_11.jpeg',
    'med_12.jpeg',
]

# 로컬 임시 폴더에 다운로드
IMAGE_DIR = '/content/images/'
os.makedirs(IMAGE_DIR, exist_ok=True)

for fname in FILENAMES:
    url  = BASE_URL + fname
    dest = os.path.join(IMAGE_DIR, fname)
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"  다운로드 완료: {fname}")
    except Exception as e:
        print(f"  다운로드 실패: {fname} → {e}")

print(f"\n이미지 폴더: {IMAGE_DIR}")

In [ ]:
# 이미지 파일 목록 확인
EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
image_paths = sorted(
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(EXTS)
)

print(f"발견된 이미지 수: {len(image_paths)}장")
for p in image_paths:
    print(' ', os.path.basename(p))

## 3. 기본 OCR — 전처리 없이 원본 이미지 텍스트 추출

In [ ]:
def ocr_basic(image_path: str) -> str:
    """원본 이미지에 대한 기본 OCR (전처리 없음)."""
    img = Image.open(image_path)
    lang   = os.environ['TESSERACT_LANG']
    config = os.environ['TESSERACT_CONFIG']
    text = pytesseract.image_to_string(img, lang=lang, config=config)
    return text


# 첫 번째 이미지로 기본 OCR 시연
sample_path = image_paths[0]

img_display = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 6))
plt.imshow(img_display)
plt.title('원본 이미지')
plt.axis('off')
plt.show()

basic_result = ocr_basic(sample_path)
print("=" * 60)
print(f"[기본 OCR 결과 — lang={os.environ['TESSERACT_LANG']}, 전처리 없음]")
print("=" * 60)
print(basic_result)

## 4. [가산점] 전처리 파이프라인으로 OCR 정확도 향상

약봉투 이미지는 배경 무늬·조명 불균일로 인식률이 낮을 수 있습니다.  
아래 5단계 전처리로 인식률을 높입니다.

| 단계 | 기법 | 목적 |
|------|------|------|
| 1 | 흑백 변환 (Grayscale) | 색상 노이즈 제거 |
| 2 | 크기 업스케일 (×2) | 작은 글자 선명화 |
| 3 | CLAHE (대비 제한 적응형 히스토그램 평탄화) | 조명 불균일 보정, 이진화 품질 향상 |
| 4 | 적응형 이진화 (Adaptive Threshold) | 국소 밝기 기준으로 컬러 배경·조명 불균일 대응 |
| 5 | 모폴로지 클로징 + 미디언 블러 | 끊어진 획 복원 및 점 노이즈 제거 |

In [ ]:
## 4. [가산점] 전처리 파이프라인으로 OCR 정확도 향상

약봉투/제품 이미지는 배경 무늬·조명 불균일·표면 텍스처로 인식률이 낮을 수 있습니다.  
3D 제품 사진의 표면 질감이 이진화 시 노이즈로 변환되는 문제까지 고려해 6단계로 구성했습니다.

| 단계 | 기법 | 목적 |
|------|------|------|
| 1 | 흑백 변환 (Grayscale) | 색상 노이즈 제거 |
| 2 | 크기 업스케일 (×2) | 작은 글자 선명화 |
| 3 | 가우시안 블러 (5×5) | 박스 표면 등 미세 텍스처 사전 제거 |
| 4 | CLAHE (clipLimit=1.5) | 조명 불균일 보정 (노이즈 증폭 최소화) |
| 5 | 적응형 이진화 (blockSize=51, C=15) | 강한 글자 엣지만 통과, 약한 텍스처 엣지 필터링 |
| 6 | 모폴로지 클로징 + 미디언 블러 | 끊어진 획 복원 및 점 노이즈 제거 |

# [가산점] 전처리 파이프라인
def preprocess(image_path: str) -> np.ndarray:
    img = cv2.imread(image_path)

    # 1단계: 흑백 변환
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2단계: 2배 업스케일 — 작은 글자 인식률 향상
    scaled = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # 3단계: 가우시안 블러 — 이진화 전 미세 텍스처(박스 표면 등) 사전 제거
    #   3D 제품 사진의 표면 질감이 이진화 시 노이즈로 변환되는 문제 방지
    smoothed = cv2.GaussianBlur(scaled, (5, 5), 0)

    # 4단계: CLAHE — 조명 불균일 보정 (clipLimit 낮춰 노이즈 증폭 억제)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    equalized = clahe.apply(smoothed)

    # 5단계: 적응형 이진화 — blockSize·C 모두 키워 약한 엣지(텍스처) 필터링
    #   blockSize=51: 넓은 영역 평균으로 임계값 계산 → 국소 텍스처에 덜 민감
    #   C=15: 강한 글자 엣지만 통과
    binary = cv2.adaptiveThreshold(
        equalized, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=51,
        C=15
    )

    # 6단계: 모폴로지 클로징 — 이진화로 끊어진 획을 복원 (한글에 특히 효과적)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    # 미디언 블러 — 점 노이즈 최종 제거
    denoised = cv2.medianBlur(closed, 3)

    return denoised


def ocr_with_preprocessing(image_path: str) -> tuple[np.ndarray, str]:
    processed = preprocess(image_path)
    pil_img = Image.fromarray(processed)
    # [가산점] 환경변수 TESSERACT_LANG + CONFIG 참조 → kor+eng + PSM/OEM 최적화
    lang   = os.environ['TESSERACT_LANG']
    config = os.environ['TESSERACT_CONFIG']
    text = pytesseract.image_to_string(pil_img, lang=lang, config=config)
    return processed, text


print("전처리 함수 정의 완료")
print(f"사용 언어  : {os.environ['TESSERACT_LANG']}")
print(f"OCR 설정   : {os.environ['TESSERACT_CONFIG']}")

In [ ]:
processed_img, preprocessed_result = ocr_with_preprocessing(sample_path)

# 시각적 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB))
axes[0].set_title('원본 이미지', fontsize=14)
axes[0].axis('off')

axes[1].imshow(processed_img, cmap='gray')
axes[1].set_title('전처리 후 (흑백 → 업스케일 → 이진화 → 노이즈 제거)', fontsize=14)
axes[1].axis('off')

plt.suptitle('전처리 전/후 비교', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# OCR 결과 비교 출력
print("=" * 60)
print("[원본 OCR 결과]")
print("=" * 60)
print(basic_result)

print("\n" + "=" * 60)
print("[전처리 후 OCR 결과]")
print("=" * 60)
print(preprocessed_result)

## 6. [가산점] 전체 이미지 배치 처리

In [ ]:
# [가산점] 약봉투 이미지 전체 배치 OCR 처리 (오류 처리 포함)
def batch_ocr(paths: list[str]) -> list[dict]:
    results = []
    for path in paths:
        try:
            _, text = ocr_with_preprocessing(path)
            results.append({'file': os.path.basename(path), 'text': text, 'error': None})
            print(f"  완료: {os.path.basename(path)}")
        except Exception as e:
            results.append({'file': os.path.basename(path), 'text': '', 'error': str(e)})
            print(f"  실패: {os.path.basename(path)} → {e}")
    return results


print("배치 OCR 시작...")
all_results = batch_ocr(image_paths)
success = sum(1 for r in all_results if r['error'] is None)
print(f"\n총 {len(all_results)}장 중 {success}장 성공")

In [ ]:
# 전체 이미지 OCR 결과 출력
for i, result in enumerate(all_results, 1):
    print(f"{'=' * 60}")
    print(f"[이미지 {i}] {result['file']}")
    print(f"{'=' * 60}")
    print(result['text'])
    print()

## 7. [가산점] 약봉투 특화 키워드 하이라이팅

추출된 텍스트에서 약품명·복약 정보·주의사항 관련 키워드를 찾아 강조 표시합니다.

In [ ]:
# [가산점] 약봉투 특화 키워드 하이라이팅
PHARMACY_KEYWORDS = [
    # 복약 관련
    '1정', '2정', '3정', '1캡슐', '1회', '2회', '3회', '1일', '2일', '3일',
    '식전', '식후', '취침전', '공복',
    # 보관 관련
    '밀폐용기', '실온보관', '냉장보관', '차광',
    # 주의 관련
    '주의', '금기', '부작용', '졸음', '음주',
    # 약효 관련
    '진통제', '소염', '항생제', '위장약', '소화',
]


def highlight_keywords(text: str, keywords: list[str]) -> str:
    """발견된 키워드를 [ ] 로 감싸 강조 표시."""
    for kw in keywords:
        text = text.replace(kw, f'[★{kw}★]')
    return text


# 첫 번째 이미지 결과에 키워드 하이라이팅 적용
sample_text = all_results[0]['text']
highlighted = highlight_keywords(sample_text, PHARMACY_KEYWORDS)

print("=" * 60)
print(f"[키워드 하이라이팅 결과] {all_results[0]['file']}")
print("=" * 60)
print(highlighted)

In [ ]:
# 전체 이미지에서 발견된 키워드 통계
from collections import Counter

keyword_counts: Counter = Counter()
for result in all_results:
    for kw in PHARMACY_KEYWORDS:
        count = result['text'].count(kw)
        if count > 0:
            keyword_counts[kw] += count

print("=" * 60)
print(f"[전체 {len(all_results)}장에서 발견된 약봉투 키워드 빈도]")
print("=" * 60)
for kw, cnt in keyword_counts.most_common():
    print(f"  {kw:10s}: {cnt}회")

# 키워드 빈도 막대 그래프
if keyword_counts:
    labels, values = zip(*keyword_counts.most_common(10))
    plt.figure(figsize=(10, 4))
    plt.bar(labels, values, color='steelblue')
    plt.title('약봉투 키워드 빈도 Top 10', fontsize=14)
    plt.xlabel('키워드')
    plt.ylabel('등장 횟수')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 8. 최종 요약

| 항목 | 내용 |
|------|------|
| OCR 엔진 | Tesseract + pytesseract |
| 언어 옵션 | `lang='kor+eng'` (환경변수 `TESSERACT_LANG`으로 관리) |
| 전처리 단계 | 흑백 → 2× 업스케일 → 적응형 이진화 → 미디언 블러 |
| 처리 이미지 수 | 약봉투 12장 배치 처리 |
| 창의적 기능 | 약봉투 특화 키워드 하이라이팅 + 빈도 시각화 |

**가산점 적용 사항 (코드 주석 표시: `# [가산점]`)**
1. `lang='kor+eng'` — 환경변수(`TESSERACT_LANG`) 기반 한글+영어 혼용 인식
2. 4단계 전처리 파이프라인 (적응형 이진화로 영어 전용 이미지도 처리)
3. 전체 이미지 배치 처리
4. 약봉투 도메인 특화 키워드 하이라이팅 및 빈도 시각화